In [1]:
taxonomy_df = spark.table(
    "demo.silver.content_taxonomy"
)

evidence_df = spark.table(
    "demo.silver.learner_concept_evidence"
)

print("Taxonomy rows:", taxonomy_df.count())
print("Evidence rows:", evidence_df.count())

Taxonomy rows: 10
Evidence rows: 34


In [2]:
from pyspark.sql.functions import col

taxonomy_required_fields_df = taxonomy_df.filter(
    col("taxonomy_id").isNull()
    | col("domain").isNull()
    | col("normalized_domain").isNull()
    | col("taxonomy_level").isNull()
    | col("source_type").isNull()
    | col("first_detected_at").isNull()
    | col("last_detected_at").isNull()
    | col("validation_status").isNull()
    | col("is_active").isNull()
    | col("taxonomy_hash").isNull()
)

print(
    "Taxonomy rows with missing required fields:",
    taxonomy_required_fields_df.count()
)

Taxonomy rows with missing required fields: 0


In [3]:
taxonomy_duplicate_ids_df = (
    taxonomy_df
    .groupBy("taxonomy_id")
    .count()
    .filter(col("count") > 1)
)

taxonomy_duplicate_hashes_df = (
    taxonomy_df
    .groupBy("taxonomy_hash")
    .count()
    .filter(col("count") > 1)
)

print(
    "Duplicate taxonomy IDs:",
    taxonomy_duplicate_ids_df.count()
)

print(
    "Duplicate taxonomy hashes:",
    taxonomy_duplicate_hashes_df.count()
)

Duplicate taxonomy IDs: 0
Duplicate taxonomy hashes: 0


In [4]:
invalid_taxonomy_values_df = taxonomy_df.filter(
    ~col("taxonomy_level").isin(
        "domain",
        "topic",
        "subtopic",
        "concept"
    )
    | ~col("validation_status").isin(
        "approved",
        "pending",
        "flagged",
        "rejected"
    )
)

print(
    "Invalid taxonomy level/status rows:",
    invalid_taxonomy_values_df.count()
)

Invalid taxonomy level/status rows: 0


In [5]:
taxonomy_with_parent_df = (
    taxonomy_df.alias("child")
    .join(
        taxonomy_df.alias("parent"),
        col("child.parent_taxonomy_id")
        == col("parent.taxonomy_id"),
        "left"
    )
    .select(
        col("child.taxonomy_id"),
        col("child.taxonomy_level"),
        col("child.domain"),
        col("child.topic"),
        col("child.subtopic"),
        col("child.concept_name"),
        col("child.parent_taxonomy_id"),
        col("parent.taxonomy_level").alias("parent_level")
    )
)

In [6]:
missing_parent_df = taxonomy_with_parent_df.filter(
    col("parent_taxonomy_id").isNotNull()
    & col("parent_level").isNull()
)

print(
    "Rows with missing parent:",
    missing_parent_df.count()
)

missing_parent_df.show(truncate=False)

Rows with missing parent: 0
+-----------+--------------+------+-----+--------+------------+------------------+------------+
|taxonomy_id|taxonomy_level|domain|topic|subtopic|concept_name|parent_taxonomy_id|parent_level|
+-----------+--------------+------+-----+--------+------------+------------------+------------+
+-----------+--------------+------+-----+--------+------------+------------------+------------+



In [7]:
invalid_parent_level_df = taxonomy_with_parent_df.filter(

    (
        (col("taxonomy_level") == "domain")
        & col("parent_taxonomy_id").isNotNull()
    )

    | (
        (col("taxonomy_level") == "topic")
        & (
            col("parent_taxonomy_id").isNull()
            | (col("parent_level") != "domain")
        )
    )

    | (
        (col("taxonomy_level") == "subtopic")
        & (
            col("parent_taxonomy_id").isNull()
            | (col("parent_level") != "topic")
        )
    )

    | (
        (col("taxonomy_level") == "concept")
        & (
            col("parent_taxonomy_id").isNull()
            | ~col("parent_level").isin(
                "topic",
                "subtopic"
            )
        )
    )
)

print(
    "Rows with invalid parent level:",
    invalid_parent_level_df.count()
)

invalid_parent_level_df.show(truncate=False)

Rows with invalid parent level: 0
+-----------+--------------+------+-----+--------+------------+------------------+------------+
|taxonomy_id|taxonomy_level|domain|topic|subtopic|concept_name|parent_taxonomy_id|parent_level|
+-----------+--------------+------+-----+--------+------------+------------------+------------+
+-----------+--------------+------+-----+--------+------------+------------------+------------+



In [8]:
evidence_required_fields_df = evidence_df.filter(
    col("evidence_id").isNull()
    | col("user_id").isNull()
    | col("taxonomy_id").isNull()
    | col("evidence_type").isNull()
    | col("evidence_time").isNull()
    | col("source_table").isNull()
    | col("processing_time").isNull()
)

print(
    "Evidence rows with missing required fields:",
    evidence_required_fields_df.count()
)

Evidence rows with missing required fields: 0


In [9]:
duplicate_evidence_ids_df = (
    evidence_df
    .groupBy("evidence_id")
    .count()
    .filter(col("count") > 1)
)

print(
    "Duplicate evidence IDs:",
    duplicate_evidence_ids_df.count()
)

Duplicate evidence IDs: 0


In [10]:
invalid_evidence_types_df = evidence_df.filter(
    ~col("evidence_type").isin(
        "practice_attempt",
        "ai_insight",
        "validated_insight",
        "pre_feedback",
        "post_feedback",
        "check_in"
    )
)

print(
    "Invalid evidence types:",
    invalid_evidence_types_df.count()
)

Invalid evidence types: 0


In [11]:
invalid_evidence_taxonomy_links_df = (
    evidence_df.alias("e")
    .join(
        taxonomy_df.select("taxonomy_id").alias("t"),
        col("e.taxonomy_id") == col("t.taxonomy_id"),
        "left_anti"
    )
)

print(
    "Invalid evidence taxonomy links:",
    invalid_evidence_taxonomy_links_df.count()
)

invalid_evidence_taxonomy_links_df.select(
    "evidence_id",
    "user_id",
    "taxonomy_id",
    "evidence_type"
).show(truncate=False)

Invalid evidence taxonomy links: 0
+-----------+-------+-----------+-------------+
|evidence_id|user_id|taxonomy_id|evidence_type|
+-----------+-------+-----------+-------------+
+-----------+-------+-----------+-------------+



In [12]:
invalid_practice_evidence_df = evidence_df.filter(
    (col("evidence_type") == "practice_attempt")
    & (
        col("attempt_id").isNull()
        | col("event_id").isNull()
        | col("is_correct").isNull()
        | col("score").isNull()
        | col("hints_used").isNull()
        | col("attempt_duration_seconds").isNull()
        | col("attempt_number").isNull()
    )
)

invalid_ai_insight_evidence_df = evidence_df.filter(
    (col("evidence_type") == "ai_insight")
    & (
        col("insight_id").isNull()
        | col("event_id").isNull()
        | col("extraction_confidence").isNull()
    )
)

invalid_validated_insight_evidence_df = evidence_df.filter(
    (col("evidence_type") == "validated_insight")
    & (
        col("validation_id").isNull()
        | col("insight_id").isNull()
        | col("event_id").isNull()
        | col("extraction_confidence").isNull()
        | col("semantic_match_score").isNull()
        | col("reliability_score").isNull()
        | col("contradiction_flag").isNull()
    )
)

print(
    "Invalid practice-attempt evidence:",
    invalid_practice_evidence_df.count()
)

print(
    "Invalid AI-insight evidence:",
    invalid_ai_insight_evidence_df.count()
)

print(
    "Invalid validated-insight evidence:",
    invalid_validated_insight_evidence_df.count()
)

Invalid practice-attempt evidence: 0
Invalid AI-insight evidence: 0
Invalid validated-insight evidence: 0


In [13]:
invalid_pre_feedback_df = evidence_df.filter(
    (col("evidence_type") == "pre_feedback")
    & (
        col("feedback_id").isNull()
        | col("confidence_score").isNull()
        | col("perceived_understanding_score").isNull()
        | col("perceived_difficulty_score").isNull()
    )
)

invalid_post_feedback_df = evidence_df.filter(
    (col("evidence_type") == "post_feedback")
    & (
        col("feedback_id").isNull()
        | col("confidence_score").isNull()
        | col("perceived_understanding_score").isNull()
        | col("perceived_difficulty_score").isNull()
        | col("still_confused").isNull()
    )
)

invalid_check_in_df = evidence_df.filter(
    (col("evidence_type") == "check_in")
    & (
        col("feedback_id").isNull()
        | col("confidence_score").isNull()
        | col("perceived_understanding_score").isNull()
        | col("still_confused").isNull()
    )
)

print(
    "Invalid pre-feedback evidence:",
    invalid_pre_feedback_df.count()
)

print(
    "Invalid post-feedback evidence:",
    invalid_post_feedback_df.count()
)

print(
    "Invalid check-in evidence:",
    invalid_check_in_df.count()
)

Invalid pre-feedback evidence: 0
Invalid post-feedback evidence: 0
Invalid check-in evidence: 0


In [14]:
invalid_evidence_ranges_df = evidence_df.filter(

    (
        col("score").isNotNull()
        & (
            (col("score") < 0)
            | (col("score") > 1)
        )
    )

    | (
        col("hints_used").isNotNull()
        & (col("hints_used") < 0)
    )

    | (
        col("attempt_duration_seconds").isNotNull()
        & (col("attempt_duration_seconds") < 0)
    )

    | (
        col("attempt_number").isNotNull()
        & (col("attempt_number") < 1)
    )

    | (
        col("extraction_confidence").isNotNull()
        & (
            (col("extraction_confidence") < 0)
            | (col("extraction_confidence") > 1)
        )
    )

    | (
        col("semantic_match_score").isNotNull()
        & (
            (col("semantic_match_score") < 0)
            | (col("semantic_match_score") > 1)
        )
    )

    | (
        col("reliability_score").isNotNull()
        & (
            (col("reliability_score") < 0)
            | (col("reliability_score") > 1)
        )
    )

    | (
        col("confidence_score").isNotNull()
        & (
            (col("confidence_score") < 1)
            | (col("confidence_score") > 10)
        )
    )

    | (
        col("perceived_understanding_score").isNotNull()
        & (
            (col("perceived_understanding_score") < 1)
            | (col("perceived_understanding_score") > 10)
        )
    )

    | (
        col("perceived_difficulty_score").isNotNull()
        & (
            (col("perceived_difficulty_score") < 1)
            | (col("perceived_difficulty_score") > 10)
        )
    )
)

print(
    "Evidence rows with invalid numeric ranges:",
    invalid_evidence_ranges_df.count()
)

invalid_evidence_ranges_df.select(
    "evidence_id",
    "evidence_type",
    "score",
    "hints_used",
    "attempt_duration_seconds",
    "attempt_number",
    "extraction_confidence",
    "semantic_match_score",
    "reliability_score",
    "confidence_score",
    "perceived_understanding_score",
    "perceived_difficulty_score"
).show(truncate=False)

Evidence rows with invalid numeric ranges: 0
+-----------+-------------+-----+----------+------------------------+--------------+---------------------+--------------------+-----------------+----------------+-----------------------------+--------------------------+
|evidence_id|evidence_type|score|hints_used|attempt_duration_seconds|attempt_number|extraction_confidence|semantic_match_score|reliability_score|confidence_score|perceived_understanding_score|perceived_difficulty_score|
+-----------+-------------+-----+----------+------------------------+--------------+---------------------+--------------------+-----------------+----------------+-----------------------------+--------------------------+
+-----------+-------------+-----+----------+------------------------+--------------+---------------------+--------------------+-----------------+----------------+-----------------------------+--------------------------+



In [15]:
print("Taxonomy rows:", taxonomy_df.count())
print("Evidence rows:", evidence_df.count())

evidence_df.groupBy(
    "evidence_type"
).count().orderBy(
    "evidence_type"
).show()

Taxonomy rows: 10
Evidence rows: 34
+-----------------+-----+
|    evidence_type|count|
+-----------------+-----+
|       ai_insight|    9|
|         check_in|    5|
|    post_feedback|    3|
| practice_attempt|    5|
|     pre_feedback|    3|
|validated_insight|    9|
+-----------------+-----+

